# **🏆 WC 26' Match Outcome Predictor**
This dataset includes 49,393 results of international football matches starting from the very first official match in 1872 up to 2025. The matches range from FIFA World Cup to FIFI Wild Cup to regular friendly matches. The matches are strictly men's full internationals and the data does not include Olympic Games or matches where at least one of the teams was the nation's B-team, U-23 or a league select team.

To build this World Cup 2026 Match Prediction Model, data will be taken strictly between 2018-2026 to reduce the variations in rule changes and other factors that many steer the predictions

### **1. Clean and Process Dataset**
Create a separate clean dataframe with the needed matches data ranging from 2018 to 2026. Only accounting for the major tournaments such as the AFC Championship, European Qualifiers, Nations League and the FIFA World Cup matches.

### **2. Feature Engineering and Team Performance Stats**
Modify the dataframe for feature engineered columns such as categorical columns for the tournament type, the winning team and important team performance statistics calculated using past match data such as the team scoring momentum, team ELO and form

### **3. Exploratory Data Analysis**
Break the prediction model development with an informetrics EDA about the World Cup and other match outcomes statistics

### **4. Machine Learning Model**
Using the dataset produced after feature engineering and additional information columns, create and compare performance of 3 different machine learning models and principles : Logistic Regression, XGBoost and Random Forests

#### **Further Implementations and Developments**
Build a monte carlo simulation to simulate the different scenarios, team qualifications and different world cup stage winners

In [1]:
# Install dependencies as needed:
# pip install kagglehub pandas
import kagglehub
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import seaborn as sns
from pathlib import Path

DATASET = "martj42/international-football-results-from-1872-to-2017"
FILES = ["former_names.csv", "goalscorers.csv", "results.csv", "shootouts.csv"]

# Download the full dataset once, then load each CSV
dataset_path = Path(kagglehub.dataset_download(DATASET))
dataframes = {name: pd.read_csv(dataset_path / name) for name in FILES}

former_names = dataframes["former_names.csv"]
goalscorers = dataframes["goalscorers.csv"]
results = dataframes["results.csv"]
shootouts = dataframes["shootouts.csv"]

print("Loaded shapes:")
for name, df in dataframes.items():
    print(f"  {name}: {df.shape}")

print("\nFirst 5 results:")
print(results.head())

Loaded shapes:
  former_names.csv: (36, 4)
  goalscorers.csv: (47821, 8)
  results.csv: (49493, 9)
  shootouts.csv: (678, 5)

First 5 results:
         date home_team away_team  home_score  away_score tournament     city  \
0  1872-11-30  Scotland   England         0.0         0.0   Friendly  Glasgow   
1  1873-03-08   England  Scotland         4.0         2.0   Friendly   London   
2  1874-03-07  Scotland   England         2.0         1.0   Friendly  Glasgow   
3  1875-03-06   England  Scotland         2.0         2.0   Friendly   London   
4  1876-03-04  Scotland   England         3.0         0.0   Friendly  Glasgow   

    country  neutral  
0  Scotland    False  
1   England    False  
2  Scotland    False  
3   England    False  
4  Scotland    False  


In [2]:
tour_categories = results['tournament'].unique()
print(tour_categories)

<StringArray>
[                                         'Friendly',
                         'British Home Championship',
                              'Évence Coppée Trophy',
                                      'Muratti Vase',
                                       'Copa Lipton',
                                       'Copa Newton',
                       'Copa Premio Honor Argentino',
                                     'Olympic Games',
                        'Copa Premio Honor Uruguayo',
                    'Far Eastern Championship Games',
 ...
                                     'Mapinduzi Cup',
                                   'Canadian Shield',
                           'Outrigger Challenge Cup',
                             'South Asian Super Cup',
                                   'CONCACAF Series',
                          'Al Ain International Cup',
              'Morocco, Capital of African Football',
                                  'Mukuru 4 Nations',
 'Diamond

In [3]:
conditions = [
    results['home_score'] > results['away_score'],
    results['away_score'] > results['home_score']
]

choices = [
    results['home_team'], 
    results['away_team']
]

# Add a winner column for match outcome column, for easier model training and further EDA
results['winner'] = np.select(conditions, choices, default='Draw')

results.shape

(49493, 10)

### **Team ELO Rating System**

In [4]:
# 1. Initialize all teams with a base rating of 1500
all_teams = set(results['home_team']).union(set(results['away_team']))
elo_dict = {team: 1500.0 for team in all_teams}

home_elos = []
away_elos = []

# 2. Loop through every match to calculate and update Elo ratings
for idx, row in results.iterrows():
    h_team = row['home_team']
    a_team = row['away_team']
    winner = row['winner']
    
    # Store current ratings before the match happens (prevents data leakage)
    h_elo = elo_dict[h_team]
    a_elo = elo_dict[a_team]
    home_elos.append(h_elo)
    away_elos.append(a_elo)
    
    # Calculate win probabilities based on rating differences
    prob_home = 1 / (1 + 10 ** ((a_elo - h_elo) / 400))
    prob_away = 1 - prob_home
    
    # Assign actual match outcomes from the perspective of each team
    if winner == h_team:
        score_home, score_away = 1.0, 0.0
    elif winner == a_team:
        score_home, score_away = 0.0, 1.0
    else:
        score_home, score_away = 0.5, 0.5  # Draw
        
    # Update the master dictionary with new ratings (K-factor of 32)
    elo_dict[h_team] += 32 * (score_home - prob_home)
    elo_dict[a_team] += 32 * (score_away - prob_away)

# 3. Add the new lists back to the main dataframe
results['home_elo'] = home_elos
results['away_elo'] = away_elos
results['elo_diff'] = results['home_elo'] - results['away_elo']

### **Team Momentum and Form Calculation**

In [5]:
def calculate_team_form(df, windows=[5, 10]):
    # Ensure dataframe is sorted chronologically and has a clean index
    df = df.sort_values('date').reset_index(drop=True)
    
    # Store unique match identifiers to map back perfectly later
    df['match_id'] = df.index
    
    # Create individual rows for every team performance
    home_side = df[['match_id', 'date', 'home_team', 'winner']].rename(columns={'home_team': 'team'})
    home_side['side'] = 'home'
    home_side['points'] = np.select(
        [home_side['winner'] == home_side['team'], home_side['winner'] == 'Draw'], 
        [3, 1], default=0
    )
    
    away_side = df[['match_id', 'date', 'away_team', 'winner']].rename(columns={'away_team': 'team'})
    away_side['side'] = 'away'
    away_side['points'] = np.select(
        [away_side['winner'] == away_side['team'], away_side['winner'] == 'Draw'], 
        [3, 1], default=0
    )
    
    # Combine into a single chronological timeline
    timeline = pd.concat([home_side, away_side]).sort_values('date').reset_index(drop=True)
    
    # Calculate forms and momentum per team timeline
    for w in windows:
        # 1. Standard rolling Form (PPG)
        timeline[f'form_{w}'] = (
            timeline.groupby('team')['points']
            .transform(lambda x: x.shift(1).rolling(w, min_periods=1).mean())
        )
        
        # 2. Weighted Momentum
        weights = np.arange(1, w + 1)
        def weighted_rolling(window_series):
            if len(window_series) == 0:
                return np.nan
            w_current = weights[-len(window_series):]
            return np.dot(window_series, w_current) / w_current.sum()

        timeline[f'momentum_{w}'] = (
            timeline.groupby('team')['points']
            .transform(lambda x: x.shift(1).rolling(w, min_periods=1).apply(weighted_rolling, raw=True))
        )
        
    # Separate back into home and away perspectives using unstack/pivot or explicit filtering
    for side in ['home', 'away']:
        side_df = timeline[timeline['side'] == side].set_index('match_id')
        for w in windows:
            df[f'{side}_form_{w}'] = df['match_id'].map(side_df[f'form_{w}'])
            df[f'{side}_momentum_{w}'] = df['match_id'].map(side_df[f'momentum_{w}'])
            
    return df.drop(columns=['match_id'])

In [6]:
results = calculate_team_form(results, windows=[5, 10])
form_cols = ['home_form_5', 'home_form_10', 'away_form_5', 'away_form_10', 'home_momentum_5', 'home_momentum_10', 'away_momentum_5', 'away_momentum_10']
results[form_cols] = results[form_cols].fillna(0)

results.tail(10)

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,winner,...,away_elo,elo_diff,home_form_5,home_momentum_5,home_form_10,home_momentum_10,away_form_5,away_momentum_5,away_form_10,away_momentum_10
49483,2026-06-30,Mexico,Ecuador,NaN,NaN,FIFA World Cup,Mexico City,Mexico,False,Draw,...,1886.838675,26.766601,3.0,3.000000,2.6,2.745455,2.0,1.866667,1.7,1.800000
49484,2026-07-01,Belgium,Senegal,NaN,NaN,FIFA World Cup,Seattle,United States,True,Draw,...,1808.826224,76.514252,2.2,2.066667,2.2,2.127273,0.8,1.133333,1.6,1.327273
49485,2026-07-01,England,DR Congo,NaN,NaN,FIFA World Cup,Atlanta,United States,True,Draw,...,1701.035442,256.661946,2.6,2.466667,2.3,2.254545,1.0,1.266667,1.5,1.418182
49486,2026-07-01,United States,Bosnia and Herzegovina,NaN,NaN,FIFA World Cup,Santa Clara,United States,False,Draw,...,1647.940872,183.600449,1.8,1.600000,1.8,1.581818,1.2,1.400000,1.5,1.309091
49487,2026-07-02,Portugal,Croatia,NaN,NaN,FIFA World Cup,Toronto,Canada,True,Draw,...,1893.879085,65.051716,2.2,1.933333,1.9,2.054545,1.8,2.200000,2.1,1.963636
49488,2026-07-02,Spain,Austria,NaN,NaN,FIFA World Cup,Inglewood,United States,True,Draw,...,1848.314474,204.712294,2.2,2.466667,2.2,2.200000,2.0,1.533333,2.0,1.890909
49489,2026-07-02,Switzerland,Algeria,NaN,NaN,FIFA World Cup,Vancouver,Canada,True,Draw,...,1837.021108,31.157036,2.2,2.333333,1.7,1.909091,2.0,1.733333,2.0,1.854545
49490,2026-07-03,Argentina,Cape Verde,NaN,NaN,FIFA World Cup,Miami Gardens,United States,True,Draw,...,1641.355547,431.257249,3.0,3.000000,3.0,3.000000,1.2,1.000000,1.2,1.054545
49491,2026-07-03,Australia,Egypt,NaN,NaN,FIFA World Cup,Arlington,United States,True,Draw,...,1774.966720,30.899346,1.0,1.066667,1.1,1.236364,1.6,1.533333,1.6,1.563636
49492,2026-07-03,Colombia,Ghana,NaN,NaN,FIFA World Cup,Kansas City,United States,True,Draw,...,1598.802997,352.573566,2.6,2.333333,2.0,2.109091,1.0,1.000000,0.8,0.781818


### **World Cup and Major International Matches Filter**

In [7]:
results['date'] = pd.to_datetime(results['date'])
recent_results = results[results['date'] >= '2018-01-01'].copy()
recent_results.head(10)

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,winner,...,away_elo,elo_diff,home_form_5,home_momentum_5,home_form_10,home_momentum_10,away_form_5,away_momentum_5,away_form_10,away_momentum_10
41297,2018-01-02,Iraq,United Arab Emirates,0.0,0.0,Gulf Cup,Kuwait City,Kuwait,True,Draw,...,1667.523725,-15.466518,2.2,2.333333,2.0,2.236364,1.6,1.600000,1.5,1.672727
41298,2018-01-02,Oman,Bahrain,1.0,0.0,Gulf Cup,Kuwait City,Kuwait,True,Oman,...,1549.943756,43.436157,2.4,2.400000,2.2,2.327273,1.8,1.666667,1.9,1.800000
41299,2018-01-05,Oman,United Arab Emirates,0.0,0.0,Gulf Cup,Kuwait City,Kuwait,True,Draw,...,1666.811935,-59.421974,2.4,2.600000,2.2,2.472727,1.8,1.400000,1.6,1.581818
41300,2018-01-07,Estonia,Sweden,1.0,1.0,Friendly,Abu Dhabi,United Arab Emirates,True,Draw,...,1771.034366,-173.615595,2.0,2.133333,2.0,1.963636,2.0,1.733333,2.0,1.781818
41301,2018-01-11,Denmark,Sweden,0.0,1.0,Friendly,Abu Dhabi,United Arab Emirates,True,Sweden,...,1763.644193,2.288908,2.2,2.066667,2.0,2.163636,1.6,1.400000,1.8,1.600000
41302,2018-01-11,Indonesia,Iceland,0.0,6.0,Friendly,Sleman,Indonesia,False,Iceland,...,1732.456911,-287.248387,2.0,1.866667,1.7,1.781818,2.0,1.533333,1.9,1.818182
41303,2018-01-11,Jordan,Finland,1.0,2.0,Friendly,Abu Dhabi,United Arab Emirates,True,Finland,...,1612.295248,-61.175713,1.8,1.800000,1.7,1.636364,2.2,2.066667,1.6,1.745455
41304,2018-01-14,Indonesia,Iceland,1.0,4.0,Friendly,Jakarta,Indonesia,False,Iceland,...,1737.597129,-297.528822,1.8,1.200000,1.4,1.472727,2.0,1.866667,2.2,2.018182
41305,2018-01-27,Moldova,South Korea,0.0,1.0,Friendly,Antalya,Turkey,True,South Korea,...,1777.080136,-401.655103,0.2,0.066667,0.6,0.327273,2.2,2.333333,1.3,1.727273
41306,2018-01-28,Yorkshire,Ellan Vannin,1.0,1.0,Friendly,Fitzwilliam,England,False,Draw,...,1534.194513,-34.194513,0.0,0.000000,0.0,0.000000,1.0,1.000000,1.4,1.054545


In [8]:
# Define a regex pattern for the specific tournaments
pattern = 'FIFA World Cup|AFC Asian Cup|UEFA Nations League'

# Filter the DataFrame using the pattern
tournament_mask = recent_results['tournament'].str.contains(pattern, case=False, na=False)
wc_results = recent_results[tournament_mask].copy().reset_index(drop=True)

wc_results = wc_results.sort_values('date').reset_index(drop=True)

## **Logistic Regression Model**

In [9]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score, accuracy_score, confusion_matrix

def logistic_reg_model(df):
    # 1. Map target string variable ('winner') to 3 discrete numeric classes
    # Class 0: Home Win, Class 1: Draw, Class 2: Away Win
    conditions = [
        df['winner'] == df['home_team'],
        df['winner'] == 'Draw',
        df['winner'] == df['away_team']
    ]
    choices = [0, 1, 2]
    df['target'] = np.select(conditions, choices, default=np.nan)
    
    # Drop rows where target couldn't be resolved or any leftover missing values
    model_df = df.dropna(subset=['target']).copy()
    model_df['target'] = model_df['target'].astype(int)
    
    # 2. Define our engineered features
    feature_cols = [
        'home_form_5', 'away_form_5', 
        'home_momentum_5', 'away_momentum_5',
        'home_form_10', 'away_form_10', 
        'home_momentum_10', 'away_momentum_10',
        'home_elo', 'away_elo', 'elo_diff'
    ]
    
    X = model_df[feature_cols]
    y = model_df['target']
    
    # 3. Train/Test Split (Chronological split is ideal for sports, 
    # but a stratified split ensures balanced class distributions)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    
    # 4. Feature Scaling (Crucial for Logistic Regression convergence)
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # 5. Initialize and train Multi-class Logistic Regression
    # 'multinomial' handles 3 classes explicitly; max_iter avoids convergence warnings
    model = LogisticRegression(solver='lbfgs', max_iter=1000, random_state=42)
    model.fit(X_train_scaled, y_train)
    
    # 6. Model Predictions
    y_pred = model.predict(X_test_scaled)
    
    # 7. Evaluate Performance
    print("         MODEL PERFORMANCE METRICS               ")
    print("==================================================")
    print(f"Overall Accuracy:     {accuracy_score(y_test, y_pred):.4f}")
    print(f"Macro F1-Score:       {f1_score(y_test, y_pred, average='macro'):.4f}")
    print(f"Weighted F1-Score:    {f1_score(y_test, y_pred, average='weighted'):.4f}")
    
    print("\nDetailed Classification Report:")
    target_names = ['Home Win (0)', 'Draw (1)', 'Away Win (2)']
    print(classification_report(y_test, y_pred, target_names=target_names))
    
    print("\nConfusion Matrix:")
    cm = confusion_matrix(y_test, y_pred)
    cm_df = pd.DataFrame(cm, index=target_names, columns=[f"Pred {n}" for n in target_names])
    print(cm_df)
    
    return model, scaler

# Run the training pipeline
trained_model, feature_scaler = logistic_reg_model(wc_results)

         MODEL PERFORMANCE METRICS               
Overall Accuracy:     0.6161
Macro F1-Score:       0.4613
Weighted F1-Score:    0.5392

Detailed Classification Report:
              precision    recall  f1-score   support

Home Win (0)       0.63      0.86      0.73       265
    Draw (1)       0.33      0.01      0.02       130
Away Win (2)       0.60      0.69      0.64       178

    accuracy                           0.62       573
   macro avg       0.52      0.52      0.46       573
weighted avg       0.55      0.62      0.54       573


Confusion Matrix:
              Pred Home Win (0)  Pred Draw (1)  Pred Away Win (2)
Home Win (0)                229              2                 34
Draw (1)                     80              1                 49
Away Win (2)                 55              0                123


## **Poisson Regression Model**

In [10]:
import statsmodels.api as sm
from scipy.stats import poisson
from sklearn.metrics import mean_absolute_error

def _outcome_probs_from_poisson(lambda_home, lambda_away, max_goals=10):
    """Convert expected goals into home-win / draw / away-win probabilities."""
    goal_range = np.arange(max_goals + 1)
    probs = np.zeros((len(lambda_home), 3))

    for i, (lh, la) in enumerate(zip(lambda_home, lambda_away)):
        p_home = poisson.pmf(goal_range, lh)
        p_away = poisson.pmf(goal_range, la)
        score_matrix = np.outer(p_home, p_away)

        probs[i, 0] = np.tril(score_matrix, k=-1).sum()  # home win
        probs[i, 1] = np.trace(score_matrix)             # draw
        probs[i, 2] = np.triu(score_matrix, k=1).sum()   # away win

    return probs


def poisson_regression_model(df, max_goals=10):
    # 1. Keep only completed matches with known scores
    model_df = df.dropna(subset=['home_score', 'away_score']).copy()
    model_df['home_score'] = model_df['home_score'].astype(int)
    model_df['away_score'] = model_df['away_score'].astype(int)

    # 2. Encode match outcome for classification evaluation
    conditions = [
        model_df['winner'] == model_df['home_team'],
        model_df['winner'] == 'Draw',
        model_df['winner'] == model_df['away_team']
    ]
    model_df['target'] = np.select(conditions, choicelist=[0, 1, 2], default=np.nan)
    model_df = model_df.dropna(subset=['target'])
    model_df['target'] = model_df['target'].astype(int)

    feature_cols = [
        'home_form_5', 'away_form_5',
        'home_momentum_5', 'away_momentum_5',
        'home_form_10', 'away_form_10',
        'home_momentum_10', 'away_momentum_10',
        'home_elo', 'away_elo', 'elo_diff'
    ]

    X = model_df[feature_cols]
    y_home = model_df['home_score']
    y_away = model_df['away_score']
    y_outcome = model_df['target']

    # 3. Same split strategy as logistic regression for fair comparison
    (
        X_train, X_test,
        y_home_train, y_home_test,
        y_away_train, y_away_test,
        y_outcome_train, y_outcome_test
    ) = train_test_split(
        X, y_home, y_away, y_outcome,
        test_size=0.2, random_state=42, stratify=y_outcome
    )

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    X_train_const = sm.add_constant(X_train_scaled)
    X_test_const = sm.add_constant(X_test_scaled)

    # 4. Fit separate Poisson GLMs for home and away goals
    home_glm = sm.GLM(
        y_home_train, X_train_const, family=sm.families.Poisson()
    ).fit()
    away_glm = sm.GLM(
        y_away_train, X_train_const, family=sm.families.Poisson()
    ).fit()

    # 5. Predict expected goals (lambda) and derive match outcomes
    lambda_home = home_glm.predict(X_test_const)
    lambda_away = away_glm.predict(X_test_const)

    outcome_probs = _outcome_probs_from_poisson(lambda_home, lambda_away, max_goals=max_goals)
    y_pred = outcome_probs.argmax(axis=1)

    # 6. Evaluate goal prediction and match outcome performance
    print("         GOAL PREDICTION METRICS               ")
    print("==================================================")
    print(f"Home Goals MAE:       {mean_absolute_error(y_home_test, lambda_home):.4f}")
    print(f"Away Goals MAE:       {mean_absolute_error(y_away_test, lambda_away):.4f}")
    print(f"Combined Goals MAE:   {mean_absolute_error(np.r_[y_home_test, y_away_test], np.r_[lambda_home, lambda_away]):.4f}")

    print("\n         MATCH OUTCOME METRICS               ")
    print("==================================================")
    print(f"Overall Accuracy:     {accuracy_score(y_outcome_test, y_pred):.4f}")
    print(f"Macro F1-Score:       {f1_score(y_outcome_test, y_pred, average='macro'):.4f}")
    print(f"Weighted F1-Score:    {f1_score(y_outcome_test, y_pred, average='weighted'):.4f}")

    print("\nDetailed Classification Report:")
    target_names = ['Home Win (0)', 'Draw (1)', 'Away Win (2)']
    print(classification_report(y_outcome_test, y_pred, target_names=target_names))

    print("\nConfusion Matrix:")
    cm = confusion_matrix(y_outcome_test, y_pred)
    cm_df = pd.DataFrame(cm, index=target_names, columns=[f"Pred {n}" for n in target_names])
    print(cm_df)

    return home_glm, away_glm, scaler, feature_cols

# Run the training pipeline
poisson_home_model, poisson_away_model, poisson_scaler, poisson_features = poisson_regression_model(wc_results)

         GOAL PREDICTION METRICS               
Home Goals MAE:       1.0157
Away Goals MAE:       0.8148
Combined Goals MAE:   0.9153

         MATCH OUTCOME METRICS               
Overall Accuracy:     0.6257
Macro F1-Score:       0.4623
Weighted F1-Score:    0.5461

Detailed Classification Report:
              precision    recall  f1-score   support

Home Win (0)       0.65      0.87      0.74       264
    Draw (1)       0.00      0.00      0.00       127
Away Win (2)       0.59      0.71      0.64       178

    accuracy                           0.63       569
   macro avg       0.41      0.53      0.46       569
weighted avg       0.48      0.63      0.55       569


Confusion Matrix:
              Pred Home Win (0)  Pred Draw (1)  Pred Away Win (2)
Home Win (0)                229              0                 35
Draw (1)                     73              0                 54
Away Win (2)                 51              0                127


/Users/raoabdul/Documents/Development/NASA-Stardance-26/.nasa/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/raoabdul/Documents/Development/NASA-Stardance-26/.nasa/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/raoabdul/Documents/Development/NASA-Stardance-26/.nasa/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_divisio

## **Match Prediction**
Run the cell below — it trains on all data, then prompts you in the output to enter home and away teams. Type `quit` to exit.

In [ ]:
import re

# ISO country codes for flag emojis
TEAM_TO_ISO = {
    'Afghanistan': 'AF', 'Albania': 'AL', 'Algeria': 'DZ', 'Angola': 'AO', 'Argentina': 'AR',
    'Armenia': 'AM', 'Australia': 'AU', 'Austria': 'AT', 'Azerbaijan': 'AZ', 'Bahrain': 'BH',
    'Bangladesh': 'BD', 'Belarus': 'BY', 'Belgium': 'BE', 'Benin': 'BJ', 'Bolivia': 'BO',
    'Bosnia and Herzegovina': 'BA', 'Botswana': 'BW', 'Brazil': 'BR', 'Bulgaria': 'BG',
    'Burkina Faso': 'BF', 'Burundi': 'BI', 'Cameroon': 'CM', 'Canada': 'CA', 'Cape Verde': 'CV',
    'Chile': 'CL', 'China PR': 'CN', 'Colombia': 'CO', 'Costa Rica': 'CR', "Côte d'Ivoire": 'CI',
    'Ivory Coast': 'CI', 'Croatia': 'HR', 'Cuba': 'CU', 'Curaçao': 'CW', 'Cyprus': 'CY',
    'Czech Republic': 'CZ', 'Czechia': 'CZ', 'DR Congo': 'CD', 'Denmark': 'DK', 'Ecuador': 'EC',
    'Egypt': 'EG', 'El Salvador': 'SV', 'England': 'GB', 'Estonia': 'EE', 'Ethiopia': 'ET',
    'Finland': 'FI', 'France': 'FR', 'Gabon': 'GA', 'Georgia': 'GE', 'Germany': 'DE', 'Ghana': 'GH',
    'Greece': 'GR', 'Guatemala': 'GT', 'Haiti': 'HT', 'Honduras': 'HN', 'Hungary': 'HU',
    'Iceland': 'IS', 'India': 'IN', 'Indonesia': 'ID', 'Iran': 'IR', 'Iraq': 'IQ', 'Ireland': 'IE',
    'Israel': 'IL', 'Italy': 'IT', 'Jamaica': 'JM', 'Japan': 'JP', 'Jordan': 'JO', 'Kazakhstan': 'KZ',
    'Kenya': 'KE', 'Kosovo': 'XK', 'Kuwait': 'KW', 'Latvia': 'LV', 'Lebanon': 'LB', 'Libya': 'LY',
    'Lithuania': 'LT', 'Luxembourg': 'LU', 'Madagascar': 'MG', 'Malaysia': 'MY', 'Mali': 'ML',
    'Malta': 'MT', 'Mexico': 'MX', 'Moldova': 'MD', 'Montenegro': 'ME', 'Morocco': 'MA',
    'Mozambique': 'MZ', 'Netherlands': 'NL', 'New Zealand': 'NZ', 'Nicaragua': 'NI', 'Nigeria': 'NG',
    'North Korea': 'KP', 'North Macedonia': 'MK', 'Northern Ireland': 'GB', 'Norway': 'NO',
    'Oman': 'OM', 'Pakistan': 'PK', 'Palestine': 'PS', 'Panama': 'PA', 'Paraguay': 'PY', 'Peru': 'PE',
    'Philippines': 'PH', 'Poland': 'PL', 'Portugal': 'PT', 'Qatar': 'QA', 'Romania': 'RO',
    'Russia': 'RU', 'Rwanda': 'RW', 'Saudi Arabia': 'SA', 'Scotland': 'GB', 'Senegal': 'SN',
    'Serbia': 'RS', 'Slovakia': 'SK', 'Slovenia': 'SI', 'South Africa': 'ZA', 'South Korea': 'KR',
    'Korea Republic': 'KR', 'Spain': 'ES', 'Sudan': 'SD', 'Sweden': 'SE', 'Switzerland': 'CH',
    'Syria': 'SY', 'Thailand': 'TH', 'Togo': 'TG', 'Trinidad and Tobago': 'TT', 'Tunisia': 'TN',
    'Turkey': 'TR', 'Türkiye': 'TR', 'Uganda': 'UG', 'Ukraine': 'UA', 'United Arab Emirates': 'AE',
    'United States': 'US', 'Uruguay': 'UY', 'Uzbekistan': 'UZ', 'Venezuela': 'VE', 'Vietnam': 'VN',
    'Wales': 'GB', 'Yemen': 'YE', 'Zambia': 'ZM', 'Zimbabwe': 'ZW',
}

FORMER_TO_CURRENT = dict(zip(former_names['former'], former_names['current']))


def _iso_to_flag(iso_code):
    return ''.join(chr(127397 + ord(c)) for c in iso_code.upper())


def team_flag(team_name):
    iso = TEAM_TO_ISO.get(team_name)
    return _iso_to_flag(iso) if iso else '🏳️'


def resolve_team_name(name, known_teams):
    name = name.strip()
    if name in known_teams:
        return name
    if name in FORMER_TO_CURRENT:
        resolved = FORMER_TO_CURRENT[name]
        if resolved in known_teams:
            return resolved
    matches = [t for t in known_teams if t.lower() == name.lower()]
    if len(matches) == 1:
        return matches[0]
    partial = [t for t in known_teams if name.lower() in t.lower()]
    if len(partial) == 1:
        return partial[0]
    raise ValueError(
        f"Could not find team '{name}'. "
        f"Try an exact name from the dataset, e.g. 'South Korea' not 'Korea Republic'."
    )


def parse_matchup(matchup):
    parts = re.split(r'\s+vs\.?\s+|\s+v\s+', matchup.strip(), maxsplit=1, flags=re.IGNORECASE)
    if len(parts) != 2 or not parts[0] or not parts[1]:
        raise ValueError("Use format: 'Home Team vs Away Team' (e.g. 'Brazil vs Japan')")
    return parts[0].strip(), parts[1].strip()


def build_team_state(df):
    """Latest ELO, form, and momentum for every team from full match history."""
    df = df.sort_values('date')
    all_teams = set(df['home_team']).union(df['away_team'])
    elo_dict = {team: 1500.0 for team in all_teams}
    stats = {
        team: {'elo': 1500.0, 'form_5': 0.0, 'form_10': 0.0, 'momentum_5': 0.0, 'momentum_10': 0.0}
        for team in all_teams
    }

    for _, row in df.iterrows():
        h_team, a_team = row['home_team'], row['away_team']
        h_elo, a_elo = elo_dict[h_team], elo_dict[a_team]

        if pd.notna(row['home_score']) and pd.notna(row['away_score']):
            prob_home = 1 / (1 + 10 ** ((a_elo - h_elo) / 400))
            winner = row['winner']
            if winner == h_team:
                score_home, score_away = 1.0, 0.0
            elif winner == a_team:
                score_home, score_away = 0.0, 1.0
            else:
                score_home, score_away = 0.5, 0.5
            elo_dict[h_team] += 32 * (score_home - prob_home)
            elo_dict[a_team] += 32 * (score_away - (1 - prob_home))

        stats[h_team] = {
            'elo': elo_dict[h_team],
            'form_5': row['home_form_5'],
            'form_10': row['home_form_10'],
            'momentum_5': row['home_momentum_5'],
            'momentum_10': row['home_momentum_10'],
        }
        stats[a_team] = {
            'elo': elo_dict[a_team],
            'form_5': row['away_form_5'],
            'form_10': row['away_form_10'],
            'momentum_5': row['away_momentum_5'],
            'momentum_10': row['away_momentum_10'],
        }

    return stats


def fit_poisson_on_all_data(df):
    """Train Poisson GLMs on the entire completed-match dataset."""
    model_df = df.dropna(subset=['home_score', 'away_score']).copy()
    model_df['home_score'] = model_df['home_score'].astype(int)
    model_df['away_score'] = model_df['away_score'].astype(int)

    feature_cols = [
        'home_form_5', 'away_form_5',
        'home_momentum_5', 'away_momentum_5',
        'home_form_10', 'away_form_10',
        'home_momentum_10', 'away_momentum_10',
        'home_elo', 'away_elo', 'elo_diff',
    ]

    X = model_df[feature_cols]
    y_home = model_df['home_score']
    y_away = model_df['away_score']

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    X_const = sm.add_constant(X_scaled, has_constant='add')

    home_glm = sm.GLM(y_home, X_const, family=sm.families.Poisson()).fit()
    away_glm = sm.GLM(y_away, X_const, family=sm.families.Poisson()).fit()

    return home_glm, away_glm, scaler, feature_cols


def _top_scorelines(lambda_home, lambda_away, top_n=5, max_goals=10):
    goal_range = np.arange(max_goals + 1)
    p_home = poisson.pmf(goal_range, lambda_home)
    p_away = poisson.pmf(goal_range, lambda_away)
    scorelines = [
        ((h, a), p_home[h] * p_away[a])
        for h in range(max_goals + 1)
        for a in range(max_goals + 1)
    ]
    scorelines.sort(key=lambda x: x[1], reverse=True)
    return scorelines[:top_n]


def predict_match(
    matchup,
    home_glm=None,
    away_glm=None,
    scaler=None,
    feature_cols=None,
    team_state=None,
    training_df=None,
    history_df=None,
    max_goals=10,
):
    """
    Predict a match from a string like 'Brazil vs Japan'.
    First team is home; trains on all data in training_df by default.
    """
    if training_df is None:
        training_df = wc_results
    if history_df is None:
        history_df = results
    if team_state is None:
        team_state = build_team_state(history_df)

    known_teams = set(team_state.keys())
    home_raw, away_raw = parse_matchup(matchup)
    home_team = resolve_team_name(home_raw, known_teams)
    away_team = resolve_team_name(away_raw, known_teams)

    if home_glm is None or away_glm is None or scaler is None or feature_cols is None:
        home_glm, away_glm, scaler, feature_cols = fit_poisson_on_all_data(training_df)

    home = team_state[home_team]
    away = team_state[away_team]

    feature_vector = pd.DataFrame([{
        'home_form_5': home['form_5'],
        'away_form_5': away['form_5'],
        'home_momentum_5': home['momentum_5'],
        'away_momentum_5': away['momentum_5'],
        'home_form_10': home['form_10'],
        'away_form_10': away['form_10'],
        'home_momentum_10': home['momentum_10'],
        'away_momentum_10': away['momentum_10'],
        'home_elo': home['elo'],
        'away_elo': away['elo'],
        'elo_diff': home['elo'] - away['elo'],
    }], columns=feature_cols)

    X_scaled = scaler.transform(feature_vector)
    X_const = sm.add_constant(X_scaled, has_constant='add')

    lambda_home = float(home_glm.predict(X_const)[0])
    lambda_away = float(away_glm.predict(X_const)[0])
    probs = _outcome_probs_from_poisson([lambda_home], [lambda_away], max_goals=max_goals)[0]
    top_scores = _top_scorelines(lambda_home, lambda_away)

    home_flag = team_flag(home_team)
    away_flag = team_flag(away_team)
    pred_home = int(round(lambda_home))
    pred_away = int(round(lambda_away))

    outcome_labels = [f'{home_flag} {home_team} Win', 'Draw', f'{away_flag} {away_team} Win']
    outcome_idx = int(np.argmax(probs))
    n_matches = len(training_df.dropna(subset=['home_score', 'away_score']))

    print()
    print("=" * 52)
    print(f"  {home_flag} {home_team}  vs  {away_flag} {away_team}")
    print("=" * 52)
    print(f"  Model: Poisson regression ({n_matches:,} training matches)")
    print("-" * 52)
    print(f"  Predicted score : {lambda_home:.2f} - {lambda_away:.2f}  (~{pred_home}-{pred_away})")
    print(f"  ELO ratings     : {home['elo']:.0f} vs {away['elo']:.0f}")
    print("-" * 52)
    print("  Outcomes")
    print(f"    {outcome_labels[0]:<28} {probs[0]*100:>6.1f}%")
    print(f"    {'Draw':<28} {probs[1]*100:>6.1f}%")
    print(f"    {outcome_labels[2]:<28} {probs[2]*100:>6.1f}%")
    print("-" * 52)
    print("  Most likely scorelines")
    print(f"    {'Score':<8} {'Probability':>12}")
    for (h, a), prob in top_scores:
        print(f"    {h}-{a:<7} {prob*100:>11.1f}%")
    print("-" * 52)
    print(f"  Most likely result: {outcome_labels[outcome_idx]} ({probs[outcome_idx]*100:.1f}%)")
    print("=" * 52)
    print()
    return {
        'home_team': home_team,
        'away_team': away_team,
        'lambda_home': lambda_home,
        'lambda_away': lambda_away,
        'home_win_prob': probs[0],
        'draw_prob': probs[1],
        'away_win_prob': probs[2],
        'top_scorelines': top_scores,
        'predicted_outcome': outcome_labels[outcome_idx],
    }


# Train once on all completed tournament matches
POISSON_HOME, POISSON_AWAY, POISSON_SCALER, POISSON_FEATURES = fit_poisson_on_all_data(wc_results)
TEAM_STATE = build_team_state(results)

print("=" * 52)
print("  MATCH PREDICTOR")
print("=" * 52)
print("  Enter teams below (first team is home).")
print("  Type 'quit' at any prompt to exit.")
print("=" * 52)

while True:
    home = input("\nHome team: ").strip()
    if home.lower() in ("quit", "q", "exit"):
        print("Exiting predictor.")
        break

    away = input("Away team: ").strip()
    if away.lower() in ("quit", "q", "exit"):
        print("Exiting predictor.")
        break

    if not home or not away:
        print("Please enter both teams.")
        continue

    try:
        predict_match(
            f"{home} vs {away}",
            home_glm=POISSON_HOME,
            away_glm=POISSON_AWAY,
            scaler=POISSON_SCALER,
            feature_cols=POISSON_FEATURES,
            team_state=TEAM_STATE,
        )
    except Exception as exc:
        print(f"\nError: {exc}\n")